In [53]:
!pip install pandas folium -q
import pandas as pd
import folium
from IPython.display import HTML, display

# ======================
# 1. Dữ liệu ban đầu
# ======================

# Vị trí ban đầu của 2 xe
vehicle_positions = {
    "V1": [10.7626, 106.6601],
    "V2": [10.7626, 106.6601]
}

# Các điểm nhu cầu
demand_points = [
    [10.7630, 106.6610],
    [10.7640, 106.6620],
    [10.7605, 106.6585],
    [10.7592, 106.6570],
    [10.7650, 106.6640]
]

# Các mốc thời gian
time_steps = [1, 2, 3, 4, 5]

# ======================
# 2. Mô phỏng di chuyển
# ======================
simulation = []

for t in time_steps:
    for vehicle_id in vehicle_positions:
        # Chọn điểm đích
        if vehicle_id == "V1":
            target = demand_points[t % len(demand_points)]
        else:
            target = demand_points[(t + 1) % len(demand_points)]

        old_lat = vehicle_positions[vehicle_id][0]
        old_lon = vehicle_positions[vehicle_id][1]

        # Di chuyển 40% quãng đường đến điểm đích
        new_lat = old_lat + (target[0] - old_lat) * 0.4
        new_lon = old_lon + (target[1] - old_lon) * 0.4

        # Cập nhật vị trí mới
        vehicle_positions[vehicle_id] = [new_lat, new_lon]

        # Lưu kết quả
        simulation.append({
            "time": t,
            "vehicle": vehicle_id,
            "lat": new_lat,
            "lon": new_lon
        })

# Chuyển sang DataFrame
sim_df = pd.DataFrame(simulation)

print("Bảng mô phỏng:")
print(sim_df)

# ======================
# 3. Vẽ bản đồ vị trí cuối cùng
# ======================
m = folium.Map(location=[10.762, 106.660], zoom_start=15)

# Vẽ điểm nhu cầu
for i, point in enumerate(demand_points):
    folium.Marker(
        point,
        popup=f"Điểm nhu cầu {i+1}",
        icon=folium.Icon(color="blue")
    ).add_to(m)

# Lấy vị trí cuối cùng của mỗi xe
last_positions = sim_df.groupby("vehicle").tail(1)

for _, row in last_positions.iterrows():
    folium.Marker(
        [row["lat"], row["lon"]],
        popup=f"{row['vehicle']} - thời gian {row['time']}",
        icon=folium.Icon(color="red")
    ).add_to(m)

display(HTML(m._repr_html_()))

Bảng mô phỏng:
   time vehicle        lat         lon
0     1      V1  10.763160  106.660860
1     1      V2  10.761760  106.659460
2     2      V1  10.762096  106.659916
3     2      V2  10.760736  106.658476
4     3      V1  10.760938  106.658750
5     3      V2  10.762442  106.660686
6     4      V1  10.762563  106.660850
7     4      V2  10.762665  106.660811
8     5      V1  10.762738  106.660910
9     5      V2  10.763199  106.661287
